<img style="float: center;" src='https://github.com/spacetelescope/jwst-pipeline-notebooks/raw/main/_static/stsci_header.png' alt="stsci_logo" width="900px"/> 

# Extracting FS Data from MOS/IFU Observations

**Authors**: Kayli Glidic (kglidic@stsci.edu), with contributions from Chris Hayes; NIRSpec branch<br>
**Last Updated**: April 30, 2026 </br>

**Purpose**:<br>
The primary goal of this notebook is to demonstrate how to extract fixed slit (FS) data from multi-object spectroscopy (MOS) and integral field unit (IFU) NIRSpec observations.

**[Data](#4.-Download-the-Data)**:<br>
This notebook is set up to use PRISM MOS and IFU observations in which the fixed slits (FS) were generally observing background. The data were obtained from Proposal IDs (PIDs) 2750 (MOS) and 1216 (IFU).


---

## Table of Contents
* [1. Introduction](#1.-Introduction)
* [2. Package Imports](#2.-Package-Imports)
* [3. Helper Functions](#3.-Helper-Functions)
* [4. Download the Data](#4.-Download-the-Data)
* [5. Prepare to Extract FS Data from MOS Observations](#5.-Prepare-to-Extract-FS-Data-from-MOS-Observations)
* [6. Prepare to Extract FS Data from IFU Observations](#6.-Prepare-to-Extract-FS-Data-from-IFU-Observations)
* [7. Extract FS Data from MOS/IFU Observations (Stage 2)](#7.-Extract-FS-Data-from-MOS/IFU-Observations-(Stage-2))
* [8. Extract FS Data from MOS/IFU Observations (Stage 3)](#8.-Extract-FS-Data-from-MOS/IFU-Observations-(Stage-3))





---

## 1. Introduction

The Near Infrared Spectrograph (NIRSpec) includes several [FS apertures](https://jwst-docs.stsci.edu/jwst-near-infrared-spectrograph/nirspec-instrumentation/nirspec-fixed-slits#gsc.tab=0:~:text=NIRSpec%20Fixed%20Slits-,NIRSpec%20Fixed%20Slits,-The%C2%A0NIRSpec) machined directly into the mounting plate of the micro-shutter assembly (MSA). 

The available NIRSpec FS's are:
* S200A1
* S200A2
* S200B1 (redundant with S200A1; not used for science)
* S400A1
* S1600A1

These slits remain open during all observations, including MOS and IFU modes. Because these FS spectra fall on a different detector region than MOS or IFU spectra, they are unaffected by failed open MSA shutters. As a result, the FS's can be used as additional background observations (if no source contamination) or have other sources of potential interest fall within them. 

Currently, however, FS data are not automatically processed from MOS or IFU observations, except when planned in APT using the FS + MOS template, which allows FS targets to be explicitly planned and processed.

This notebook provides a workaround for extracting and processing FS data from these types of observations.



---

## 2. Package Imports

In [ ]:
# General imports.
import os
import glob
import numpy as np
import itertools
from collections import defaultdict

# Astropy imports.
from astropy.table import Table
from astropy.io import fits
from astropy.stats import sigma_clip
from astroquery.mast import Observations

# JWST pipeline imports.
from jwst import datamodels  # JWST pipeline utilities: datamodels.
from jwst.associations import asn_from_list as afl  # Tools for creating association files.
from jwst.associations.lib.rules_level3_base import DMS_Level3_Base  # Define Lvl3 ASN.
from jwst.pipeline import Spec2Pipeline  # calwebb_spec2
from jwst.pipeline import Spec3Pipeline  # calwebb_spec3
from jwst.pipeline import Detector1Pipeline #calwebb_det1

# Plotting.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from astropy.visualization import ImageNormalize, ManualInterval, LogStretch
from astropy.visualization import LinearStretch, AsinhStretch, simple_norm

---

## 3. Helper Functions

In this section, we define a set of functions that handle the following tasks: downloading data products from MAST, adding FS targets to MSA metadata files, creating Stage 3 association files, and plotting NIRSpec spectra.


In [ ]:
# Helper function to download JWST files from MAST.
def download_jwst_files(filenames,
                        download_dir,
                        mast_dir='mast:jwst/product'):
    """
    Helper function to download JWST files from MAST.

    Parameters:
    ----------
    filenames: list of str
        List of filenames to download.
    download_dir: str
        Directory where the files will be downloaded.
    mast_dir: str
        MAST directory containing JWST products.

    Returns:
    -------
    downloaded_files: list of str
        List of downloaded file paths.
    """
    # Download data.
    downloaded_files = []
    os.makedirs(download_dir, exist_ok=True)
    for filename in filenames:
        filename = os.path.basename(filename)
        mast_path = os.path.join(mast_dir, filename)
        local_path = os.path.join(download_dir, filename)
        if os.path.exists(local_path):
            print(local_path, 'EXISTS')
        else:
            # Can let this command check if local file exists.
            # However, it will delete it if it's there
            # and the wrong size (e.g., reprocessed).
            Observations.download_file(mast_path, local_path=local_path)
        downloaded_files.append(local_path)

    return downloaded_files

In [ ]:
def add_fs_target(shutter_table,
                  source_table,
                  visit_rate_file,
                  slit_name,
                  ra, dec,
                  stellarity=1.0,
                  new_source_id=None):
    """
    Add a fixed-slit target to the MSA metadata file in the SHUTTER_INFO and SOURCE_INFO tables.
    Assumes the source is centered in the slit.

    Parameters
    ----------
    shutter_table : astropy.table.Table
        Table from the SHUTTER_INFO extension.
    source_table : astropy.table.Table
        Table from the SOURCE_INFO extension.
    visit_rate_file : str
        Path to a rate file from the visit.
    slit_name : str
        Fixed slit name.
    ra : float
        Source RA.
    dec : float
        Source DEC.
    stellarity : float, optional
        Source stellarity value. Use 1.0 for a point source and 0.0 forfPRIDTPTS
        an extended source. Default is 1.0.
    new_source_id : int, optional
        Source ID to use. If not provided, the next available source ID
        is assigned.

    Returns
    -------
    new_slit_id : int
        Newly assigned slitlet ID.
    new_source_id : int
        Source ID used for the new source.
    shutter_table : astropy.table.Table
        Updated SHUTTER_INFO table.
    source_table : astropy.table.Table
        Updated SOURCE_INFO table.
    """

    # Read visit-specific metadata from the rate file header.
    header = fits.getheader(visit_rate_file)
    msa_meta_id = header['MSAMETID']
    program = header['PROGRAM']
    n_dithers = header['NUMDTHPT']
    
    # Create a fixed slit column in the table if it doesn't exist.
    if 'fixed_slit' not in shutter_table.colnames:
        shutter_table['fixed_slit'] = [''] * len(shutter_table)

    # Check if this fixed slit already exists for this visit.
    existing = (
        (shutter_table['fixed_slit'] == slit_name) &
        (shutter_table['msa_metadata_id'] == msa_meta_id)
    )

    if np.any(existing):
        print(f"{slit_name} already exists for msa_metadata_id={msa_meta_id}. Skipping.")
        existing_slit_id = shutter_table['slitlet_id'][existing][0]
        existing_source_id = shutter_table['source_id'][existing][0]
        return existing_slit_id, existing_source_id, shutter_table, source_table

    # Use the next available source ID if one is not provided.
    if new_source_id is None:
        new_source_id = np.max(source_table['source_id']) + 1
    
    # Use the next available slitlet ID.
    new_slit_id = np.max(shutter_table['slitlet_id']) + 1

    # Add one shutter row for each dither point.
    for dither in range(1, n_dithers+1):
        shutter_table.add_row({
            'dither_point_index': dither,
            'msa_metadata_id': msa_meta_id,
            'background': 'N',
            'estimated_source_in_shutter_x': 0.5,
            'estimated_source_in_shutter_y': 0.5,
            'primary_source': 'Y',
            'shutter_column': 0,
            'shutter_quadrant': 0,
            'shutter_row': 0,
            'shutter_state': 'OPEN',
            'slitlet_id': new_slit_id,
            'source_id': new_source_id,
            'fixed_slit': slit_name
        })

    # Add the source row.
    if new_source_id not in source_table['source_id']:
        source_table.add_row({
            'program': program,
            'source_id': new_source_id,
            'source_name': f'{program}_{new_source_id}',
            'alias': f'{new_source_id}',
            'ra': ra,
            'dec': dec,
            'preimage_id': 'None',
            'stellarity': stellarity
        })

    return new_slit_id, new_source_id, shutter_table, source_table

In [ ]:
def writel3asn(scifiles, dir='./'):
    """
    Create a Level 3 association file.

    Parameters
    ----------
    scifiles : list of str
        List of all science exposure files.
    dir : str, optional 
        Save directory.
    
    Returns
    -------
    None.
    """
    # Filter based on GRATING/FILTER.
    from collections import defaultdict
    grouped = defaultdict(lambda: {'sci': [], 'bg': []})

    for f in scifiles:
        k = (fits.getval(f, 'PROGRAM'), fits.getval(f, 'OBSERVTN'), fits.getval(f, 'FILTER'), fits.getval(f, 'GRATING'))
        grouped[k]['sci'].append(f)

    # Make ASN for each FILTER/GRATING.
    for (pid, obs, filt, grat), files in grouped.items():
        name = f"Program{pid}_{obs}_{filt}_{grat}".lower()
        asnfile = os.path.join(dir, f"{name}_l3asn.json")
        filenames = [os.path.basename(f) for f in files['sci']]
        asn = afl.asn_from_list(filenames, rule=DMS_Level3_Base, product_name=name)

        with open(asnfile, 'w') as f:
            f.write(asn.dump()[1])
    print("Level 3 ASN creation complete!")

In [ ]:
def display_spectra(spectra,
                    compare_x1d=None,
                    compare_mast=None,
                    integration=None,
                    extname='data',
                    source_id=None,
                    source_unit='FLUX',
                    expand_wavelength_gap=True,
                    plot_resample=True,
                    plot_errors=False,
                    cmap='viridis',
                    bad_color=(1, 0.7, 0.7),
                    aspect='auto',
                    vmin=None,
                    vmax=None,
                    scale='asinh',
                    title_prefix=None,
                    title_path=False,
                    y_limits=None,
                    is_stage3=False):

    """
    Display 2D and 1D spectra (Stage 2/3).

    Parameters
    ----------
    spectra : list of str
        A list of data products (e.g., CAL, S2D, X1D files).
    compare_x1d : list of str, optional
        A list of 1D spectra for comparison (X1D files).
    compare_mast : list of str, optional
        A list of 1D spectra from MAST for comparison (X1D files).
    integration : {None, 'min', int}, optional
        Specifies the integration to use for multi-integration data.
        If 'min', the minimum value across all integrations is used.
        If an integer, the specific integration index is used (default 0).
     extname : str, optional
        The name of the data extension to extract ('data', 'dq', etc.).
    source_id : int or none, optional
        Identifier for the source/slit to be displayed. Default is None.
    source_unit : str, optional
        Override data source units ('POINT' == 'FLUX' or 'EXTENDED' = 'SURF_BRIGHT').
    expand_wavelength_gap : bool, optional
        If True, expands gaps in the wavelength data for better visualization.
    plot_resample : bool, optional
        If True, plots resampled (S2D) data products;
        otherwise, plots calibrated (CAL) data. Default is True.
    plot_errors : bool, optional
        If True, plots the error bands for the 1D spectra. Default is False.
    cmap : str, optional
        Colormap to use for displaying the images. Default is 'viridis'.
    bad_color : tuple of float, optional
        Color to use for bad pixels. Default is light red (1, 0.7, 0.7).
    aspect : str, optional
        Aspect ratio of the plot. Default is 'auto'.
    vmin : float, optional
        Minimum value for color scaling. If None, determined from the data.
    vmax : float, optional
        Maximum value for color scaling. If None, determined from the data.
    scale : {'linear', 'log', 'asinh'}, optional
        Scale to use for the image normalization. Default is 'asinh'.
    title_prefix : str, optional
        Optional prefix for the plot title.
    title_path : bool, optional
        If True, uses the full file path for the title;
        otherwise, uses the basename. Default is False.
    y_limits : tuple of float, optional
        Limits for the y-axis of the 1D spectrum plot.
        If None, limits are determined from the data.
    is_stage3 : bool, optional
        Plot stage 3 products? Default is False.

    Returns
    -------
    None.
    """

    # ---------------------------------Check Inputs---------------------------------
    spectra = [spectra] if isinstance(spectra, str) else spectra
    compare_x1d = [compare_x1d] if isinstance(compare_x1d, str) else compare_x1d
    compare_mast = [compare_mast] if isinstance(compare_mast, str) else compare_mast

    # Assign a default source_id if one was not supplied.
    if source_id is None:
        ftype = "cal"
        if plot_resample:
            ftype = "s2d"
        for file in spectra:
            if ftype in file:
                source_id = datamodels.open(file)[0].slits[0].source_id
                break

    src_str = str(source_id)

    # Plot stage 3 products?
    if is_stage3:

        
         # Stage 3 products: plot files if source_id is in filename OR SLTNAME in header
        def filter_prod(products, source_id):
            """Filter products based on filename OR header SLTNAME."""
            filtered = []
            for f in products:
                header = fits.getheader(f, ext=1)
                sltname = header.get('SLTNAME', '').lower() if header else ''
                if str(source_id).lower() in os.path.basename(f).lower() or sltname == str(source_id).lower():
                    filtered.append(f)
            return filtered
        #def filter_prod(products, source_id):
        #    """Filter products based on the source_id."""
#
        #    return [
        #        f for f in products
        #        if src_str.lower() in f and ('FXD_SLIT' not in fits.getheader(f, ext=0) or fits.getheader(f, ext=0)['FXD_SLIT'].lower() == src_str.lower())]

        spectra = filter_prod(spectra, source_id)
        compare_x1d = filter_prod(compare_x1d, source_id) if compare_x1d else None
        compare_mast = filter_prod(compare_mast, source_id) if compare_mast else None

    ftypes = {ftype: [f for f in spectra
                      if ftype in f] for ftype in ["cal", "s2d", "x1d"]}
    products = sorted(ftypes['s2d']) if plot_resample else sorted(ftypes['cal'])
    if not products:
        raise ValueError("No valid data products found for plotting.")

    # --------------------------------Set up figures-------------------------------
    total_plots = len(products) + bool(ftypes['x1d'])
    height_ratios = [1] * len(products) + ([3] if bool(ftypes['x1d']) else [])
    fig, axes = plt.subplots(total_plots, 1, figsize=(15, 5 * total_plots),
                             sharex=False, height_ratios=height_ratios)
    fig.subplots_adjust(hspace=0.2, wspace=0.2)
    ax2d, ax1d = (axes[:-1], axes[-1]) if bool(ftypes['x1d']) else (axes, None)

    cmap = plt.get_cmap(cmap)  # Set up colormap and bad pixel color.
    cmap.set_bad(bad_color, 1.0)
    colors = plt.get_cmap('tab10').colors
    color_cycle = itertools.cycle(colors)

    # ---------------------------------Plot spectra--------------------------------
    for i, product in enumerate(products):
        model = datamodels.open(product)  # Open files as JWST datamodels.

        # Extract the correct 2D source spectrum if there are multiple.
        slit_m = model
        if 'slits' in model:
            slits = model.slits
            slit_m = next((s for s in slits
                           if getattr(s, 'name', None) == source_id), None)
            slit_m = slit_m or next((s for s in model.slits
                                     if s.source_id == source_id), None)
            if not slit_m:
                print(f"'{source_id}' not found/invalid.")
                print(f"Available source_ids: {[s.source_id for s in slits][:5]}")
                break

        # Check if 'fixed_slit' exists, otherwise fall back to 'slitlet_id'
        slit_name = (f"SLIT: {getattr(slit_m, 'name', None) or slit_m.slitlet_id}, "
                     f"SOURCE: {getattr(slit_m, 'source_id', '')}")

        # -----------------------Extract the 2D/3D data----------------------
        data_2d = getattr(slit_m, extname)
        if data_2d.ndim == 3:  # Handle multi-integration data.
            if integration == 'min':
                data_2d = np.nanmin(data_2d, axis=0)
            elif isinstance(integration, int) and 0 <= integration < data_2d.shape[0]:
                data_2d = data_2d[integration]
            else:
                raise ValueError(f"Invalid integration '{integration}' for 3D data.")

        # -----------Convert from pixels to wavelength (x-axis)--------------
        wcsobj = slit_m.meta.wcs  # Obtaining the WCS object from the meta data.
        y, x = np.mgrid[:slit_m.data.shape[0], :slit_m.data.shape[1]]
        # Coordinate transform from detector space (pixels) to sky (RA, DEC).
        det2sky = wcsobj.get_transform('detector', 'world')
        ra, dec, s2dwave = det2sky(x, y)  # RA/Dec, wavelength (microns) for each pixel.
        s2dwaves = s2dwave[0, :]  # Single row since this is the rectified spectrum.
        x_arr = np.arange(0, slit_m.data.shape[1], int(len(slit_m.data[1]) / 4))
        wav = np.round(s2dwaves[x_arr], 2)  # Populating the wavelength array.
        ax2d[i].set_xticks(x_arr, wav)

        # xticks = np.arange(np.ceil(wave_1d[0]), wave_1d[-1], 0.2)
        # xtick_pos = np.interp(xticks, wave_1d, np.arange(num_waves))
        # ax1d.set_xticks(xtick_pos)
        # ax1d.set_xticklabels([f'{xtick:.1f}' for xtick in xticks])

        # ---------------------------Scale the data-------------------------
        sigma_clipped_data = sigma_clip(np.ma.masked_invalid(data_2d), sigma=5, maxiters=3)
        vmin = np.nanmin(sigma_clipped_data) if vmin is None else vmin
        vmax = np.nanmax(sigma_clipped_data) if vmax is None else vmax
        stretch_map = {'log': LogStretch(), 'linear': LinearStretch(),
                       'asinh': AsinhStretch()}
        if scale in stretch_map:
            norm = ImageNormalize(sigma_clipped_data,
                                  interval=ManualInterval(vmin=vmin, vmax=vmax),
                                  stretch=stretch_map[scale])
        else:
            norm = simple_norm(sigma_clipped_data, vmin=vmin, vmax=vmax)

        # -------------------------Plot 1D Spectra-------------------------
        for k, (prods_1d, prefix) in enumerate([(sorted(ftypes['x1d']),
                                                 f'{title_prefix} '),
                                                (compare_x1d, 'RE-EXTRACTION '),
                                                (compare_mast, 'MAST ')]):
            if prods_1d:
                model_1d = datamodels.open(prods_1d[i])
                specs = model_1d.spec
                spec = next((s for s in specs if
                             getattr(s, 'name', None) == source_id), None)
                spec = spec or next((s for s in specs
                                     if s.source_id == source_id), None)

                if spec:
                    tab = spec.spec_table
                    wave = tab.WAVELENGTH
                    flux = tab.FLUX if source_unit == 'FLUX' else tab.SURF_BRIGHT
                    errs = tab.FLUX_ERROR if source_unit == 'FLUX' else tab.SB_ERROR

                    # Expand the array to visualize the wavelength gap.
                    if expand_wavelength_gap:
                        dx1d_wave = wave[1:] - wave[:-1]
                        igap = np.argmax(dx1d_wave)
                        dx_replace = (dx1d_wave[igap - 1] + dx1d_wave[igap + 1]) / 2.
                        nfill = int(np.round(np.nanmax(dx1d_wave) / dx_replace))

                        if nfill > 1:
                            print(nfill)
                            print(f"Expanding wavelength gap {wave[igap]:.2f} "
                                  f"-- {wave[igap + 1]:.2f} μm")

                            wave_fill = np.mgrid[wave[igap]:wave[igap + 1]:(nfill + 1) * 1j]
                            wave = np.concatenate([wave[:igap + 1],
                                                   wave_fill[1:-1],
                                                   wave[igap + 1:]])

                            if prefix != 'RE-EXTRACTION ':
                                num_rows, num_waves = data_2d.shape
                                fill_2d = np.zeros(shape=(num_rows, nfill - 1)) * np.nan
                                data_2d = np.concatenate([data_2d[:, :igap + 1],
                                                          fill_2d, data_2d[:, igap + 1:]],
                                                         axis=1)

                            fill = np.zeros(shape=(nfill - 1)) * np.nan
                            flux = np.concatenate([flux[:igap + 1], fill, flux[igap + 1:]])
                            errs = np.concatenate([errs[:igap + 1], fill, errs[igap + 1:]])
                    else:
                        nfill = 0

                    # ----------------Construct legends and annotations-----------------
                    detector = slit_m.meta.instrument.detector
                    ffilter = slit_m.meta.instrument.filter
                    grating = slit_m.meta.instrument.grating
                    dither = model.meta.dither.position_number
                    label_2d = f'{grating}/{ffilter}'
                    label_1d = f'{detector} ({grating}/{ffilter})'
                    if not is_stage3:
                        label_2d = f'Dither/Nod {dither} ({label_2d})'
                        label_1d = (f'{prefix} Dither/Nod {dither} {label_1d}')
                    else:
                        label_1d = f'{prefix}{label_1d}'
                    ax2d[i].annotate(label_2d, xy=(1, 1), xycoords='axes fraction',
                                     xytext=(-10, -10), textcoords='offset points',
                                     bbox=dict(boxstyle="round,pad=0.3",
                                               edgecolor='white',
                                               facecolor='white', alpha=0.8),
                                     fontsize=12, ha='right', va='top')

                    title_2d = (f"{title_prefix + ' ' if title_prefix else ''}"
                                f"{model.meta.filename} | {slit_name}")
                    if integration:
                        title_2d = title_2d.replace('.fits', f'[{integration}].fits')
                    ax2d[i].set_title(title_2d, fontsize=14)
                    if not bool(ftypes['x1d']):
                        ax2d[i].set_xlabel("Wavelength (μm)", fontsize=12)
                    ax2d[i].set_ylabel("Pixel Row", fontsize=12)
                    # ax2d[i].legend(fontsize=12)

                    # ------------------------------------------------------------------

                    num_waves = len(wave)
                    color = next(color_cycle)
                    ax1d.step(wave, flux, lw=1, label=label_1d, color=color)
                    if plot_errors:
                        ax1d.fill_between(np.arange(num_waves), flux - errs,
                                          flux + errs, color='grey', alpha=0.3)
                    ax1d.legend(fontsize=12)
                    ax1d.set_title(f"{title_prefix + ' ' if title_prefix else ''}"
                                   f"Extracted 1D Spectra | {slit_name}", fontsize=14)
                    ax1d.set_ylabel("Flux (Jy)" if source_unit == 'FLUX'
                                    else "Surface Brightness (MJy/sr)", fontsize=12)
                    ax1d.set_xlabel("Wavelength (μm)", fontsize=12)

                    ax1d.set_ylim(y_limits or (np.nanpercentile(flux, 1),
                                               np.nanpercentile(flux, 99.5)))

                    # --------------------Plot the 2D spectra & colorbar---------------
                    plt.subplots_adjust(left=0.05, right=0.85)
                    if k == 0:
                        im = ax2d[i].imshow(data_2d, origin='lower',
                                            cmap=cmap, norm=norm,
                                            aspect=aspect, interpolation='nearest')
                        units = slit_m.meta.bunit_data
                        cbar_ax = fig.add_axes([ax2d[i].get_position().x1 + 0.02,
                                                ax2d[i].get_position().y0, 0.02,
                                                ax2d[i].get_position().height])
                        cbar = fig.colorbar(im, cax=cbar_ax)
                        cbar.set_label(units, fontsize=12)

                    # ----------------------Add extraction region---------------------
                    ystart, ystop, xstart, xstop = (spec.extraction_ystart - 1,
                                                    spec.extraction_ystop - 1,
                                                    spec.extraction_xstart - 1,
                                                    spec.extraction_xstop - 1)
                    extract_width = ystop - ystart + 1
                    box = Rectangle((xstart, ystart), xstop - xstart + nfill,
                                    extract_width, fc='None', ec=color,
                                    lw=2, label=prefix)
                    ax2d[i].add_patch(box)
                    ax2d[i].legend()

---

## 4. Download the Data

Below, we will download example MOS and IFU data, along with the corresponding MSA metadata file for the MOS observation. Users can also set optionally set `demo` to `False` provide other rate data.

In [ ]:
# Define data directory.
data_dir = 'data/'
stage2_dir = os.path.join(data_dir, 'stage2/')
stage3_dir = os.path.join(data_dir, 'stage3/')

os.makedirs(data_dir, exist_ok=True)
os.makedirs(stage2_dir, exist_ok=True)
os.makedirs(stage3_dir, exist_ok=True)

In [ ]:
demo = True
if demo == True:
    # Download the demo example MOS and IFU rate files.
    mos_rates = [f'jw02750002001_07101_{i:05d}_nrs1_rate.fits' for i in range(1, 4)]  # 3 NODS
    ifu_rates = [f'jw01216005001_03103_{i:05d}_nrs1_rate.fits' for i in range(1, 9)]  # 8 DITHERS
    mos_rates = download_jwst_files(mos_rates, data_dir)
    ifu_rates = download_jwst_files(ifu_rates, data_dir)
else:
    # Provide list of paths to other rate files.
    mos_rates = []
    ifu_rate = []

In [ ]:
# Get the MSA metadata file name from one of the rate headers and download.
msa_metafile_name = fits.getval(mos_rates[0], 'MSAMETFL')
msa_metafile = download_jwst_files([msa_metafile_name], data_dir)[0]
msa_metafile

---

## 5. Prepare to Extract FS Data from MOS Observations

The NIRSpec MSA metadata file is a crucial component of the calibration processing for MOS exposures. It contains all the slitlet, shutter, and source configuration information required by the `calwebb_spec2` pipeline to process a MOS observation accurately. For more information on the contents of these MSA metafiles, refer to this example [notebook](https://github.com/spacetelescope/jdat_notebooks/blob/main/notebooks/NIRSpec/msa_metafile/NIRSpec_MOS_MSA_metafile.ipynb).

To process FS data from a MOS observation, the FS targets must also be included in the metadata file. In this section, we demonstrate how to edit a metadata file to include each of the FSs.

For this example we are extracting background. So, we assume that the target is centered in each slit, the sources are extended (stellarity = 0), and they are defined as the `primary_source` in the metadata file so that they are properly processed. 


<div class="alert alert-block alert-warning">

For point sources, the `pathloss` and `wavecorr` steps depend on accurate source positioning. Users should carefully inspect their data to ensure the source location is correct when performing FS extraction.

</div>

In [ ]:
# Load MSA metafile and its source and shutter tables.
msa_hdu_list = fits.open(msa_metafile)
source_table = Table(msa_hdu_list['SOURCE_INFO'].data)
shutter_table = Table(msa_hdu_list['SHUTTER_INFO'].data)

In [ ]:
# Source coordinates and stellarity.
ra = 0.0
dec = 0.0
stellarity = 0.0  # point source = 1.0, extended source = 0.0

# Add each FS to the metadata file.
for slit_name in ['S200A1', 'S200A2', 'S400A1', 'S1600A1']:
    new_slit_id, new_source_id, shutter_table, source_table = add_fs_target(
        shutter_table,
        source_table,
        mos_rates[0],  # Use one of the rate files for general header information.
        slit_name,
        ra, dec,
        stellarity=stellarity
    )

msa_hdu_list['SHUTTER_INFO'] = fits.table_to_hdu(shutter_table)
msa_hdu_list['SOURCE_INFO'] = fits.table_to_hdu(source_table)

msa_hdu_list[2].name = 'SHUTTER_INFO'
msa_hdu_list[3].name = 'SOURCE_INFO'

msa_hdu_list.writeto(msa_metafile, overwrite=True)
msa_hdu_list.close()

In [ ]:
# Load the MSA metafile and inspect additions.
msa_hdu_list = fits.open(msa_metafile)
source_table = Table(msa_hdu_list['SOURCE_INFO'].data)
shutter_table = Table(msa_hdu_list['SHUTTER_INFO'].data)
shutter_table

---

## 6. Prepare to Extract FS Data from IFU Observations

Unlike MOS observations, IFU observations do not require an MSA metadata file for processing spectra through `calwebb_spec2`. In fact, even the MOS observations do not require a metafile if the goal is simply to extract FS data. To enable FS extraction from IFU or MOS observations, the `EXP_TYPE` keyword in the rate file header must be set to `NRS_FIXEDSLIT`, which triggers the pipeline to perform FS spectral extraction. By default, the pipeline treats all FS’s as extended sources and extracts a spectrum from the center of each slit. This is ideal for getting additional background observations. **However, if attempting to process a point source, there are several important considerations**.

* To process a point source, a primary slit must be defined using the `FXD_SLIT` header keyword, and the source type must be set to point (`SRCTYAPT` = `POINT`); otherwise, the pipeline will continue to treat the source as extended. In addition, for the `wavecorr` step to run properly on the selected slit, the correct slit-dependent reference values (`V2_REF`, `V3_REF`, and `V3I_YANG`) must be provided. If these are not updated, the pipeline will retain the original IFU values and may skip this step with a warning about non-invertible corrections.

* The `XOFFSET` and `YOFFSET` values in IFU data are based on their respective dither/nod patterns and do not nessesarily correspond to FS dither/nod positions. As a result, the source may fall within the slit for some exposures but not others. Because the pipeline uses these offsets to estimate the source location, it can incorrectly place the source outside the FS, leading to steps like pathloss being skipped or applied incorrectly and the `extract_1d` step to perform poorly.


In [ ]:
# Choose a primary FS to process (optional).
FXD_SLIT = None  # 'S200A1' or S200A2 or S400A1 or S1600A1

for ifu_rate in ifu_rates:
    with fits.open(ifu_rate, 'update') as hdul:

        hdul[0].header['EXP_TYPE'] = 'NRS_FIXEDSLIT'
        #hdul[0].header['SRCTYAPT'] = 'POINT'  # To process primary source as a point source.
        
        if FXD_SLIT == 'S200A1':

            hdul[0].header['FXD_SLIT'] = FXD_SLIT  # Set the primary slit.
            #hdul[0].header['XOFFSET'] = 0.0  # Set for point source but with caution.
            #hdul[0].header['YOFFSET'] = 0.0  # Set for point source but with caution.

            # Reference postions for S200A1 FS observations.
            hdul[1].header['V2_REF'] = 331.867035  # [arcsec] Telescope V2 coord of reference point
            hdul[1].header['V3_REF'] = -479.417542  # [arcsec] Telescope V3 coord of reference point 
            hdul[1].header['VPARITY'] = -1  # Relative sense of rotation between Ideal xy and
            hdul[1].header['V3I_YANG'] = 138.84190369  # [deg] Angle from V3 axis to Ideal y axis

        elif FXD_SLIT == 'S200A2':

            hdul[0].header['FXD_SLIT'] = FXD_SLIT 
            #hdul[0].header['XOFFSET'] = 0.0 
            #hdul[0].header['YOFFSET'] = 0.0

            # Reference postions for S200A2 FS observations.
            hdul[1].header['V2_REF'] = 314.697998
            hdul[1].header['V3_REF'] = -489.614227 
            hdul[1].header['VPARITY'] = -1
            hdul[1].header['V3I_YANG'] = 138.91368103

        elif FXD_SLIT == 'S400A1':

            hdul[0].header['FXD_SLIT'] = FXD_SLIT 
            #hdul[0].header['XOFFSET'] = 0.0 
            #hdul[0].header['YOFFSET'] = 0.0

            # Reference postions for S400A1 FS observations.
            hdul[1].header['V2_REF'] = 321.59079
            hdul[1].header['V3_REF'] = -478.107422
            hdul[1].header['VPARITY'] = -1
            hdul[1].header['V3I_YANG'] = 138.864151

        elif FXD_SLIT == 'S1600A1':

            hdul[0].header['FXD_SLIT'] = FXD_SLIT 
            #hdul[0].header['XOFFSET'] = 0.0 
            #hdul[0].header['YOFFSET'] = 0.0

            # Reference postions for S1600A1 FS observations.
            hdul[1].header['V2_REF'] = 321.26297
            hdul[1].header['V3_REF'] = -473.851288
            hdul[1].header['VPARITY'] = -1
            hdul[1].header['V3I_YANG'] = 138.85295105

---

## 7. Extract FS Data from MOS/IFU Observations (Stage 2)

Run Stage 2 and extract the spectra. Note we are not providing Stage 2 association files here.

In [ ]:
# Select a FS's source to extract and inspect.
slit_names = ['S200A1', 'S200A2', 'S400A1', 'S1600A1']

In [ ]:
# Set up a dictionary to define how the Spec2 pipeline should be configured.

spec2dict = defaultdict(dict)
 
# Extract specific sources; saves on processing time.
if slit_names is not None:
    spec2dict['extract_2d']['slit_names'] = slit_names

# We assume the source is centered in the slit.
spec2dict['extract_1d']['use_source_posn'] = True


In [ ]:
# Run Stage 2 pipeline using the custom spec2dict dictionary.

for file in mos_rates + ifu_rates:
    print(f"Applying Stage 2 Corrections & Calibrations to: {file}")
    spec2sci_result = Spec2Pipeline.call(file,
                                        save_results=True,
                                        steps=spec2dict,
                                        output_dir=stage2_dir)
            
    print("Stage 2 has been completed! \n")


In [ ]:
# List the Stage 2 products.

# -----------------------------Science files-----------------------------
sci_cal = sorted(glob.glob(stage2_dir + '*_cal.fits'))
sci_s2d = sorted(glob.glob(stage2_dir + '*_s2d.fits'))
sci_x1d = sorted(glob.glob(stage2_dir + '*_x1d.fits'))

print(f"SCIENCE | Stage 2 CAL Products:\n{'-' * 20}\n" + "\n".join(sci_cal))
print(f"SCIENCE | Stage 2 S2D Products:\n{'-' * 20}\n" + "\n".join(sci_s2d))
print(f"SCIENCE | Stage 2 X1D Products:\n{'-' * 20}\n" + "\n".join(sci_x1d))

In [ ]:
# Source units can be FLUX or SURF_BRIGHT.
display_spectra(sci_s2d + sci_x1d,
                source_id='S400A1', source_unit='SURF_BRIGHT', scale='log',
                vmin=0, vmax=3, #y_limits=(-1e-6, 0.3e-5),
                title_prefix='REPROCESSED', is_stage3=False)

---

## 8. Extract FS Data from MOS/IFU Observations (Stage 3)

Run Stage 3 and extract the spectra. We create Stage 3 association files here to show what a final stage 3 FS products look like.

In [ ]:
# Set up a dictionary to define how the Spec3 pipeline should be configured.
spec3dict = defaultdict(dict)
spec3dict['extract_1d']['use_source_posn'] = True

In [ ]:
# Write ASN files and proccess through Stage 3.
writel3asn(sci_cal, dir=stage2_dir)
for spec3_asn in glob.glob(stage2_dir+'*l3asn.json'):
        print(f"Applying Stage 3 Corrections & Calibrations to: "f"{os.path.basename(spec3_asn)}")
        spec3_result = Spec3Pipeline.call(spec3_asn,
                                          save_results=True,
                                          steps=spec3dict,
                                          output_dir=stage3_dir)
print("Stage 3 has been completed! \n")

In [ ]:
# List the Stage 3 products.

stage3_cal = sorted(glob.glob(stage3_dir + '*_cal.fits'))
stage3_s2d = sorted(glob.glob(stage3_dir + '*_s2d.fits'))
stage3_x1d = sorted(glob.glob(stage3_dir + '*_x1d.fits'))

print(f"Stage 3 CAL Products:\n{'-' * 20}\n" + "\n".join(stage3_cal))
print(f"Stage 3 S3D Products:\n{'-' * 20}\n" + "\n".join(stage3_s2d))
print(f"Stage 3 X1D Products:\n{'-' * 20}\n" + "\n".join(stage3_x1d))

In [ ]:
# Source units can be FLUX or SURF_BRIGHT.
display_spectra(stage3_s2d + stage3_x1d,
                source_id='S400A1', source_unit='SURF_BRIGHT', scale='log',
                vmin=0, vmax=3, #y_limits=(-1e-6, 0.3e-5),
                title_prefix='REPROCESSED', is_stage3=True)

If these final Stage 3 products are background, these can later be used in master background subtraction. For information on how to define different 1D extraction regions, refer to the notebooks listed below.

---
## Related Notebooks


* [NIRSpec MOS MSA Metafile](https://github.com/spacetelescope/jdat_notebooks/blob/main/notebooks/NIRSpec/msa_metafile/NIRSpec_MOS_MSA_metafile.ipynb)
* [NIRSpec Pipeline Notebooks](https://github.com/spacetelescope/jwst-pipeline-notebooks/tree/main/notebooks/NIRSPEC)

---

<figure>
       <img src="https://github.com/spacetelescope/jwst-pipeline-notebooks/blob/main/_static/stsci_footer.png?raw=true" alt="Space Telescope Logo\" align="right" style="width: 200px"/>
</figure>
   
[Top of Page](#Extracting-FS-Data-from-MOS/IFU-Observations)